## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os 
from langchain.chat_models import init_chat_model

model = init_chat_model('groq:openai/gpt-oss-120b')

response = model.invoke("What is the capital of France?")
response


AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks a simple factual question: "What is the capital of France?" The answer is Paris. Provide concise answer.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 78, 'total_tokens': 122, 'completion_time': 0.091698509, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.00311433, 'prompt_tokens_details': None, 'queue_time': 0.245210008, 'total_time': 0.094812839}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a191f3-8571-760c-8646-109f57af1ba8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 44, 'total_tokens': 122, 'output_token_details': {'reasoning': 26}})

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location"""
    return f"It's sunny in {location}"
    
model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What is the weather in Tokyo?")
print(response)
for tool_call in response.tool_calls:
    # View the tool call
    print(f"Tool call: {tool_call['name']}")
    print(f"Tool call arguments: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'The user asks for weather in Tokyo. We can call function get_weather with location "Tokyo".', 'tool_calls': [{'id': 'fc_36f79f56-ddec-492e-86a6-33b17c234b88', 'function': {'arguments': '{"location":"Tokyo"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 128, 'total_tokens': 175, 'completion_time': 0.09913408, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.00506125, 'prompt_tokens_details': None, 'queue_time': 0.376125938, 'total_time': 0.10419533}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a7bb72c24', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a191f3-885f-7184-ab36-20a66287044a-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Tokyo'}, 'id': 'fc_36f79f56-ddec-492e-86a6-33b17c234b88', 'type': 'tool_call'}] invalid_tool_calls=[] 

### Tool Execution Loops

In [9]:
#Step 1: Model Generates Tool Calls
messages = [{"role": "user", "content": "What is the weather in Tokyo?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute Tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool call with generated args
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass result to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Tokyo is currently sunny.
